In [ ]:
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 데이터 확인하기 2025.11.21
# 이상인 컬럼 제거 후 RandomForest 기본 모델 돌리기
import pandas as pd
import numpy  as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler # 데이터 전처리용
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix, recall_score, precision_score
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

import importlib
from utils import preprocessing

# 모듈 reload
importlib.reload(preprocessing)
# importlib.reload(user_utils)

from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns
from utils.user_utils    import get_model_train_eval
from utils.model_utils   import save_model, load_model

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

# train = pd.read_csv("../data/train.csv")
# test  = pd.read_csv("../data/test.csv")

In [13]:
# data loading
train, test = load_data()

In [14]:
# 데이터 할당
X_features, y_labels = split_features_target(train) # ID와 TARGET 모두 제거하고 X_train 만들기
X_test     = test.drop(columns=['ID'], axis=1) # test 데이터에서도 ID 제거
X_features['var3'] = X_features['var3'].replace(-999999, 2)
# var3 의 최소값 -99999 를 최빈값으로 변경하기

In [15]:
# 학습/테스트 데이터 분리
X_train, X_val, y_train, y_val = data_split(
  X_features,
  y_labels,
)

In [16]:
# 2) Scaler 생성 (train에만 fit)
scaler = StandardScaler()
scaler.fit(X_train)

,copy,True
,with_mean,True
,with_std,True


In [17]:
# 3) train, val, test에 동일한 scaler 적용
X_train_scaled = scaler.transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [18]:
# 레이블의 분포 확인
cust_cnt = y_labels.value_counts()
print(cust_cnt) # 1이 불만족 3008명, 만족이 73012

# 불만족고객의 비율
cust_rate = cust_cnt[1] / cust_cnt.sum()
print(f'불만족 고객 비율: {cust_rate:.2f}')

TARGET
0    73012
1     3008
Name: count, dtype: int64
불만족 고객 비율: 0.04


In [19]:
class ThresholdModel:
    def __init__(self, base_model, threshold):
        self.base_model = base_model
        self.threshold = threshold

    def fit(self, X, y):
        # 재학습 방지 — 이미 fit된 모델 그대로 사용
        return self

    def predict(self, X):
        proba = self.base_model.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.base_model.predict_proba(X)

In [23]:
# ================================
# 1) HyperOpt Objective 함수
# ================================
def objective_lgbm(params):

    # 정수형 파라미터 변환
    params['n_estimators']   = int(params['n_estimators'])
    params['max_depth']      = int(params['max_depth'])
    params['num_leaves']     = int(params['num_leaves'])
    params['min_child_samples'] = int(params['min_child_samples'])

    # 모델 선언
    model = LGBMClassifier(
        **params,
        random_state=0,
        n_jobs=-1,
        verbose=-1
    )

    # 학습
    model.fit(X_train, y_train)

    # 기본 Threshold = 0.5
    proba = model.predict_proba(X_val)[:, 1]
    pred = (proba >= 0.5).astype(int)

    # 스코어 계산
    f1 = f1_score(y_val, pred)

    return {'loss': -f1, 'status': STATUS_OK}


In [24]:
# ================================
# 2) HyperOpt 탐색 공간
# ================================
search_space_lgbm = {
    'n_estimators':     hp.quniform('n_estimators', 200, 800, 50),
    'learning_rate':    hp.loguniform('learning_rate', -4, -1.5),    # 0.018~0.223
    'max_depth':        hp.quniform('max_depth', 3, 10, 1),
    'num_leaves':       hp.quniform('num_leaves', 16, 60, 4),
    'min_child_samples': hp.quniform('min_child_samples', 5, 30, 1),
    'subsample':        hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
    'class_weight':     hp.choice('class_weight', [None, {0:1, 1:2}, {0:1, 1:3}, {0:1, 1:4}])
}

In [25]:
# ================================
# 3) HyperOpt 실행
# ================================
trials_lgbm = Trials()
best_lgbm_params = fmin(
    fn=objective_lgbm,
    space=search_space_lgbm,
    algo=tpe.suggest,
    max_evals=40,  # LGBM은 빠르니까 40~60까지 괜찮음
    trials=trials_lgbm,
    rstate=np.random.default_rng(42)
)

100%|██████████| 40/40 [01:05<00:00,  1.64s/trial, best loss: -0.25087719298245614]


In [28]:
# ================================
# 4) 정수형 값 변환
# ================================

# class_weight 후보 리스트
class_weight_options = [
    None,
    {0:1, 1:2},
    {0:1, 1:3},
    {0:1, 1:4}
]

# HyperOpt에서 가져온 index → 실제 dict로 매핑
class_weight_idx = best_lgbm_params['class_weight']
best_lgbm_params['class_weight'] = class_weight_options[class_weight_idx]

# 나머지 int 변환
best_lgbm_params['n_estimators']     = int(best_lgbm_params['n_estimators'])
best_lgbm_params['max_depth']        = int(best_lgbm_params['max_depth'])
best_lgbm_params['num_leaves']       = int(best_lgbm_params['num_leaves'])
best_lgbm_params['min_child_samples']= int(best_lgbm_params['min_child_samples'])

print("\n======= LGBM HyperOpt 결과(best params) =======")
print(best_lgbm_params)


======= LGBM HyperOpt 결과(best params) =======
{'class_weight': {0: 1, 1: 4}, 'colsample_bytree': np.float64(0.993915676646864), 'learning_rate': np.float64(0.05861943746307052), 'max_depth': 6, 'min_child_samples': 17, 'n_estimators': 350, 'num_leaves': 16, 'subsample': np.float64(0.6649208086751857)}


In [ ]:
# ================================
# 5) Best 모델 학습
# ================================
lgbm_best = LGBMClassifier(
    **best_lgbm_params,
    random_state=0,
    n_jobs=-1
)

lgbm_best.fit(X_train, y_train)
proba_val = lgbm_best.predict_proba(X_val)[:,1]
pred_val = (proba_val > 0.5).astype(int)

get_model_train_eval(lgbm_best, "LGBM_100_HP_lr0.5_max6_min17_est350_num16_class1vs4",
                     X_train, X_val,
                     y_train, y_val)

best_f1 = 0
best_thr = 0
thresholds = np.arange(0.01, 0.50, 0.01)

for thr in thresholds:
    pred_thr = (proba_val >= thr).astype(int)
    f1 = f1_score(y_val, pred_thr)
    if f1 > best_f1:
        best_f1 = f1
        best_thr = thr

print(f"\nBest Threshold = {best_thr:.2f}, Best F1 = {best_f1:.4f}")

lgbm_thr = ThresholdModel(lgbm_best, best_thr)

print("\n===== Threshold 적용 후 성능 =====")
get_model_train_eval(lgbm_thr,
                     f"LGBM_100_HP_lr0.5_max6_min17_est350_num16_class1vs4_thr_{best_thr:.2f}",
                     X_train, X_val, y_train, y_val)


✓ 모델 저장 완료: ../models\LGBM_100_HP_lr0.5_max6_min17_est350_num16_class1vs4.pkl
  파일 크기: 0.64 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8475, 정확도: 0.9438, 정밀도: 0.2658, 재현율: 0.2375, F1: 0.2509
오차행렬:
[[14207   395]
 [  459   143]]
실행 시간: 1.2549998760223389

Best Threshold = 0.44, Best F1 = 0.3032

===== Threshold 적용 후 성능 =====
✓ 모델 저장 완료: ../models\LGBM_100_HP_lr0.5_max6_min17_est350_num16_class1vs4_thr_0.44.pkl
  파일 크기: 0.64 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8475, 정확도: 0.9244, 정밀도: 0.2388, 재현율: 0.4153, F1: 0.3032
오차행렬:
[[13805   797]
 [  352   250]]
실행 시간: 0.13399314880371094


In [ ]:
# ================================
# 6) Threshold 최적화
# ================================


In [ ]:
# ================================
# 7) Threshold 적용 모델 생성
# ================================
